In [10]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, t
import scipy.stats as stats

df = pd.read_csv("online_shoppers_intention.csv")

buyers = df[df['Revenue'] == True]
non_buyers = df[df['Revenue'] == False]

In [11]:
alpha = 0.05

In [12]:
features = [
    "PageValues",
    "BounceRates",
    "ExitRates",
    "ProductRelated_Duration",
    "ProductRelated"
]

In [13]:
results = []

for col in features:
    t_stat, p_val = ttest_ind(
        buyers[col],
        non_buyers[col],
        equal_var=False
    )
    
    results.append([col, t_stat, p_val])
    
    print(f"\n{col} T-test")
    print("T-statistic:", round(t_stat,4))
    print("P-value:", round(p_val,6))
    
    if p_val < alpha:
        print("Result: Reject Null Hypothesis (Significant Difference)")
    else:
        print("Result: Fail to Reject Null Hypothesis")

results_df = pd.DataFrame(results, columns=["Feature", "T-statistic", "P-value"])
print("\nSummary Table:\n", results_df)


PageValues T-test
T-statistic: 31.1992
P-value: 0.0
Result: Reject Null Hypothesis (Significant Difference)

BounceRates T-test
T-statistic: -34.8464
P-value: 0.0
Result: Reject Null Hypothesis (Significant Difference)

ExitRates T-test
T-statistic: -44.3321
P-value: 0.0
Result: Reject Null Hypothesis (Significant Difference)

ProductRelated_Duration T-test
T-statistic: 14.447
P-value: 0.0
Result: Reject Null Hypothesis (Significant Difference)

ProductRelated T-test
T-statistic: 14.0017
P-value: 0.0
Result: Reject Null Hypothesis (Significant Difference)

Summary Table:
                    Feature  T-statistic        P-value
0               PageValues    31.199175  9.610670e-174
1              BounceRates   -34.846360  2.587228e-253
2                ExitRates   -44.332130   0.000000e+00
3  ProductRelated_Duration    14.446989   2.173791e-45
4           ProductRelated    14.001714   8.834785e-43


In [14]:
print("\nEffect Size (Cohen's d):")

def cohens_d(x1, x2):
    n1, n2 = len(x1), len(x2)
    
    s1, s2 = np.var(x1, ddof=1), np.var(x2, ddof=1)
    
    # pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1)*s1 + (n2 - 1)*s2) / (n1 + n2 - 2))
    
    d = (np.mean(x1) - np.mean(x2)) / pooled_std
    
    # small sample correction (Hedges g)
    correction = 1 - (3 / (4*(n1+n2) - 9))
    
    return d * correction


for col in features:
    d = cohens_d(buyers[col], non_buyers[col])
    
    print(f"{col}: Cohen's d = {round(d,4)}")
    
    if abs(d) < 0.2:
        print("  → Negligible effect")
    elif abs(d) < 0.5:
        print("  → Small effect")
    elif abs(d) < 0.8:
        print("  → Medium effect")
    else:
        print("  → Large effect")


Effect Size (Cohen's d):
PageValues: Cohen's d = 1.5648
  → Large effect
BounceRates: Cohen's d = -0.4214
  → Small effect
ExitRates: Cohen's d = -0.5852
  → Medium effect
ProductRelated_Duration: Cohen's d = 0.4262
  → Small effect
ProductRelated: Cohen's d = 0.4439
  → Small effect


In [15]:
print("\nConfidence Intervals (Mean Difference):")

for col in features:
    
    x1 = buyers[col]
    x2 = non_buyers[col]
    
    mean_diff = np.mean(x1) - np.mean(x2)
    
    se = np.sqrt(
        np.var(x1, ddof=1)/len(x1) +
        np.var(x2, ddof=1)/len(x2)
    )
    
    ci = stats.t.interval(
        0.95,
        df=len(x1) + len(x2) - 2,
        loc=mean_diff,
        scale=se
    )
    
    print(f"{col}: 95% CI = {ci}")


Confidence Intervals (Mean Difference):
PageValues: 95% CI = (np.float64(23.699713810744132), np.float64(26.877327043907545))
BounceRates: 95% CI = (np.float64(-0.021336362258778458), np.float64(-0.019063796856000518))
ExitRates: 95% CI = (np.float64(-0.029053307825212215), np.float64(-0.02659289671412585))
ProductRelated_Duration: 95% CI = (np.float64(696.8342469210547), np.float64(915.6093645478686))
ProductRelated: 95% CI = (np.float64(16.766268232487423), np.float64(22.224782990795692))


In [17]:
summary = []

for col in features:
    t_stat, p_val = ttest_ind(buyers[col], non_buyers[col], equal_var=False)
    d = cohens_d(buyers[col], non_buyers[col])
    
    summary.append([col, p_val, abs(d)])

summary_df = pd.DataFrame(summary, columns=["Feature", "P_value", "Abs_Cohen_d"])
summary_df = summary_df.sort_values(by="Abs_Cohen_d", ascending=False)

print(summary_df)

                   Feature        P_value  Abs_Cohen_d
0               PageValues  9.610670e-174     1.564762
2                ExitRates   0.000000e+00     0.585157
4           ProductRelated   8.834785e-43     0.443913
3  ProductRelated_Duration   2.173791e-45     0.426231
1              BounceRates  2.587228e-253     0.421365
